In [ ]:
import pandas as pd
from pathlib import Path
import xml.etree.ElementTree as ET

base_dir = Path("260308_darkcurrent_gifs/sort_for_waviness")
labels_dir = Path("260308_darkcurrent_gifs/labels")

image_dirs = [base_dir / "straight", base_dir / "wavy"]

rows = []

for img_dir in image_dirs:
    for img_path in img_dir.glob("*.jpg"):
    
        fname = img_path.name

        print(fname)
        orbit_type = fname[2]            # g or t
        date = fname[3:11]               # 20090812
        time = fname[12:18]              # 153213
        
        xml_name = fname.replace("_fifthfromlast.jpg", ".xml")
        xml_path = labels_dir / xml_name
        
        temperature = None
        print(xml_path)
        if xml_path.exists():
            print("yes")
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()

                for temp in root.iter():
                    if temp.tag.endswith("Device_Temperature"):
                        device = temp.find(".//{*}device_id")
                        if device is not None and device.text == "DETECTOR":
                            val = temp.find(".//{*}temperature_value")
                            if val is not None:
                                temperature = float(val.text)
                            break
            except Exception as e:
                print(f"Error reading {xml_path}: {e}")

        rows.append({
            "filename": fname,
            "type": orbit_type,
            "date": date,
            "time": time,
            "temperature_K": temperature,
            "folder": img_dir.name
        })

df = pd.DataFrame(rows)


In [ ]:
df.to_csv("darkfield_temp_date.csv")

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
df['color_pattern'] = df['folder']=='straight'

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [ ]:
df["time"] = pd.to_datetime(df["time"].str.zfill(6), format="%H%M%S")

In [ ]:
df['time'].iloc[1]

In [ ]:
t.dt.hour

In [ ]:
t = df['time']
df["seconds"] = t.dt.hour*3600 + t.dt.minute*60 + t.dt.second

In [ ]:
plt.scatter(df['seconds'],df['temperature_K'],c=df['color_pattern'],s=8,cmap="Paired")
plt.xticks(rotation=90)
plt.xlabel("time of day, seconds")
plt.ylabel("Temp, K")


In [ ]:
df[df['temperature_K'].between(155, 160)]

In [ ]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from PIL import Image

df_sorted = df[df["type"] == "g"].sort_values("temperature_K")

frames = []

counter = 0
for path in df_sorted["filename"]:
        
    parts = path.split('_')
    fitspath = parts[0]+'_'+parts[1]+'_flat.fits'
    fits_name = parts[1] + ".fits"
    print(fitspath)
    with fits.open("260308_darkcurrent_gifs/flat_fits/"+fitspath) as hdul:
        data = hdul[0].data  
        
        if counter == 0: 
            lo, hi = np.percentile(data, (2, 97))

        counter += 1 

        browse = data[-5] 

        stretched = np.clip((browse - lo) / (hi - lo), 0, 1)

        browse_uint8 = (stretched * 255).astype(np.uint8)

        img = Image.fromarray(browse_uint8, mode="L")
        
        frames.append(img)


gif_path = "sortedtemp_5thtolast_longerframes.gif"
frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=80,  
        loop=0
    )
        

In [ ]:
from PIL import ImageDraw, ImageFont

frames = []

counter = 0
for _, row in df_sorted.iterrows():
    path = row["filename"]
    temp = row["temperature_K"]  

    parts = path.split('_')
    fitspath = parts[0]+'_'+parts[1]+'_flat.fits'
    fits_name = parts[1] + ".fits"
    
    with fits.open("260308_darkcurrent_gifs/flat_fits/"+fitspath) as hdul:
        data = hdul[0].data  
        
        if counter == 0: 
            lo, hi = np.percentile(data, (2, 97))
        counter += 1 

        browse = data[-5] 
        stretched = np.clip((browse - lo) / (hi - lo), 0, 1)
        browse_uint8 = (stretched * 255).astype(np.uint8)

        img = Image.fromarray(browse_uint8, mode="L")

        draw = ImageDraw.Draw(img)
        font = ImageFont.load_default() 
        text = f"{temp:.1f} K"
        
        draw.text((5, 5), text, font=font, fill=255) 

        frames.append(img)

gif_path = "sortedtemp_5thtolast_text.gif"
frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=70, 
        loop=0
)

In [ ]:
from PIL import ImageDraw, ImageFont

frames = []

counter = 0
for _, row in df_sorted.iterrows():
    path = row["filename"]
    temp = row["temperature_K"]

    parts = path.split('_')
    fitspath = parts[0]+'_'+parts[1]+'_flat.fits'
    
    with fits.open("260308_darkcurrent_gifs/flat_fits/"+fitspath) as hdul:
        data = hdul[0].data  
        
        if counter == 0: 
            lo, hi = np.percentile(data, (2, 97))
        counter += 1 

        browse = data[-5] 
        stretched = np.clip((browse - lo) / (hi - lo), 0, 1)
        browse_uint8 = (stretched * 255).astype(np.uint8)

        img = Image.fromarray(browse_uint8, mode="L").convert("RGB")

        draw = ImageDraw.Draw(img)
        font = ImageFont.load_default()
        text = f"{temp:.1f} K"
        
        draw.text((5, 5), text, font=font, fill=(255, 0, 0))
        frames.append(img)

gif_path = "sortedtemp_5thtolast_longerframes_redtext.gif"
frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=70,   
        loop=0
)